# Pipeline 3 - Donor Churn Risk System

## 1) Problem Framing

**Goal:** Score each monetary donor on probability of churn (inactive status) and write actionable outputs.

**Hybrid decisioning strategy:**
- Deterministic rule tiers drive immediate intervention (`Critical/High/Medium/Low`).
- ML probability score provides additional risk signal and improves with more historical outcomes.

**Primary metric:** ROC-AUC (with recall-sensitive threshold discussion due to high false-negative cost).

## 2) Data Acquisition & Preparation

This notebook uses production-aligned ETL/feature modules so notebook behavior matches inference behavior:
- `donor_churn.etl.build_training_frame`
- `donor_churn.features.build_donor_feature_frame`
- `donor_churn.artifacts.*` for model persistence


In [1]:
from __future__ import annotations

import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import AdaBoostClassifier, ExtraTreesClassifier, GradientBoostingClassifier, RandomForestClassifier, StackingClassifier
from sklearn.feature_selection import RFECV, VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

# Ensure imports work regardless of notebook launch directory.
cwd = Path.cwd().resolve()
candidates = [cwd] + list(cwd.parents)
for p in candidates:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

# Also include any direct ml/ child path for local-style imports.
for p in candidates:
    ml_path = p / "ml"
    if ml_path.exists() and str(ml_path) not in sys.path:
        sys.path.insert(0, str(ml_path))

try:
    from ml.donor_churn.artifacts import save_metadata, save_metrics, save_model_bundle
    from ml.donor_churn.etl import build_training_frame
except ModuleNotFoundError:
    from donor_churn.artifacts import save_metadata, save_metrics, save_model_bundle
    from donor_churn.etl import build_training_frame

RANDOM_STATE = 42
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

frame = build_training_frame()
frame = frame.dropna(axis=0, subset=["churned"]).copy()

X = frame.drop(columns=["churned"])
y = frame["churned"].astype(int)

print("Rows:", len(frame), "| Churn rate:", round(float(y.mean()), 4))
print("Columns:", len(X.columns))


Rows: 57 | Churn rate: 0.2456
Columns: 22


## 3) Exploration

Document these confirmed findings from donor analysis in your report section:
- Recurring donors give materially higher lifetime value than non-recurring donors.
- Recurring donors have shorter average donation gaps.
- Donation frequency is strongly associated with recurring conversion.
- Acquisition channel differences are meaningful (church and social channels stand out).
- First-donation amount has weak relationship to long-term behavior.

## 4) Feature Selection (Ch. 16 style)

Planned sequence:
1. Near-zero variance filter (`VarianceThreshold`)
2. Univariate ranking against churn target
3. RFECV for compact subset selection
4. Permutation importance on best model

Given small sample size, prefer parsimonious feature sets and regularized models.


In [2]:
# Basic feature filtering + train/test split
X_model = X.drop(columns=["supporter_id"], errors="ignore").copy()

selector = VarianceThreshold(threshold=0.0)
X_selected = selector.fit_transform(X_model)
selected_cols = X_model.columns[selector.get_support()].tolist()
X_selected_df = pd.DataFrame(X_selected, columns=selected_cols, index=X_model.index)

X_train, X_test, y_train, y_test = train_test_split(
    X_selected_df,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train/Test:", X_train.shape, X_test.shape)

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE)),
    ]),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "KNN": Pipeline([("scaler", StandardScaler()), ("clf", KNeighborsClassifier())]),
    "SVM": Pipeline([("scaler", StandardScaler()), ("clf", SVC(probability=True, random_state=RANDOM_STATE))]),
    "Naive Bayes": GaussianNB(),
    "Random Forest": RandomForestClassifier(random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "AdaBoost": AdaBoostClassifier(random_state=RANDOM_STATE),
    "Extra Trees": ExtraTreesClassifier(random_state=RANDOM_STATE),
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    results.append(
        {
            "model": name,
            "roc_auc": roc_auc_score(y_test, proba),
            "f1": f1_score(y_test, pred, zero_division=0),
            "accuracy": accuracy_score(y_test, pred),
        }
    )

results_df = pd.DataFrame(results).sort_values("roc_auc", ascending=False)
results_df


Train/Test: (45, 21) (12, 21)


,model,roc_auc,f1,accuracy
6,Gradient Boosting,0.851852,0.400000,0.750000
8,Extra Trees,0.814815,0.666667,0.833333
5,Random Forest,0.796296,0.666667,0.833333
1,Decision Tree,0.666667,0.500000,0.666667
7,AdaBoost,0.666667,0.000000,0.666667
0,Logistic Regression,0.629630,0.545455,0.583333
4,Naive Bayes,0.629630,0.444444,0.583333
2,KNN,0.462963,0.000000,0.750000
3,SVM,0.296296,0.000000,0.750000


In [3]:
# Optional compact RFECV pass with logistic baseline
rfecv_model = LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE)
rfecv = RFECV(estimator=rfecv_model, step=1, cv=cv, scoring="roc_auc", min_features_to_select=5)
rfecv.fit(StandardScaler().fit_transform(X_train), y_train)

rfecv_cols = X_train.columns[rfecv.support_].tolist()
print("RFECV selected:", len(rfecv_cols))
print(rfecv_cols)

# Stacking model (using selected top candidates)
stack = StackingClassifier(
    estimators=[
        ("lr", Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE))])),
        ("rf", RandomForestClassifier(random_state=RANDOM_STATE)),
        ("svm", Pipeline([("scaler", StandardScaler()), ("clf", SVC(probability=True, random_state=RANDOM_STATE))])),
    ],
    final_estimator=LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE),
)
stack.fit(X_train[rfecv_cols], y_train)
stack_proba = stack.predict_proba(X_test[rfecv_cols])[:, 1]

print("Stack ROC-AUC:", round(roc_auc_score(y_test, stack_proba), 4))


RFECV selected: 6
['monetary_total', 'gap_trend', 'amount_trend', 'campaign_response_rate', 'acquisition_channel_Website', 'relationship_type_PartnerOrganization']
Stack ROC-AUC: 0.7407


In [4]:
# 5-8) Model selection, evaluation, and artifact persistence
best_model_name = results_df.iloc[0]["model"]
best_model = models[best_model_name]
best_model.fit(X_train, y_train)

proba = best_model.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

roc_auc = roc_auc_score(y_test, proba)
f1 = f1_score(y_test, pred, zero_division=0)
acc = accuracy_score(y_test, pred)

print("Best model:", best_model_name)
print("ROC-AUC:", round(roc_auc, 4), "F1:", round(f1, 4), "ACC:", round(acc, 4))
print(classification_report(y_test, pred, zero_division=0))

# Persist artifacts for infer.py compatibility
feature_list = X_train.columns.tolist()
scaler = None
if isinstance(best_model, Pipeline) and "scaler" in best_model.named_steps and "clf" in best_model.named_steps:
    scaler = best_model.named_steps["scaler"]
    model_to_save = best_model.named_steps["clf"]
else:
    model_to_save = best_model

save_model_bundle(model=model_to_save, scaler=scaler, feature_list=feature_list)
save_metadata(
    model_type=best_model_name,
    feature_list=feature_list,
    train_rows=len(X_train),
    test_rows=len(X_test),
    total_rows=len(frame),
)
save_metrics(roc_auc=roc_auc, f1=f1, accuracy=acc, classification_report=None)

print("Saved model artifacts to models/donor-churn/")


Best model: Gradient Boosting
ROC-AUC: 0.8519 F1: 0.4 ACC: 0.75
              precision    recall  f1-score   support

           0       0.80      0.89      0.84         9
           1       0.50      0.33      0.40         3

    accuracy                           0.75        12
   macro avg       0.65      0.61      0.62        12
weighted avg       0.72      0.75      0.73        12

Saved model artifacts to models/donor-churn/
